# Customer Intelligence System

## Section 1: Install Required Libraries

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn xgboost --quiet

## Section 2: Upload and Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

In [ ]:
from google.colab import files
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)
print(df.shape)
df.head()

In [ ]:
df.info()
df.describe()

## Section 3: Data Cleaning

In [ ]:
df.columns = df.columns.str.strip()

df.drop_duplicates(inplace=True)

numeric_cols = df.columns.drop('country')
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

print("Missing values after cleaning:")
print(df.isnull().sum())
print("\nShape:", df.shape)
df.head()

## Section 4: Feature Isolation and Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

country_names = df['country'].values
X = df.drop(columns=['country'])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Feature matrix shape:", X_scaled.shape)
print("Features:", list(X.columns))

## Section 5: Elbow Method

In [ ]:
from sklearn.cluster import KMeans

k_range = range(2, 11)
inertia_values = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia_values.append(kmeans.inertia_)
    print(f"k={k}, Inertia={kmeans.inertia_:.2f}")

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(k_range, inertia_values, 'o-', linewidth=2, markersize=8)
plt.axvline(x=3, color='red', linestyle='--', label='k=3')
plt.title('Elbow Method')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.xticks(list(k_range))
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Section 6: K-Means Model (best_k = 3)

In [ ]:
best_k = 3

kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
kmeans_labels = kmeans_final.fit_predict(X_scaled)

df['KMeans_Cluster'] = kmeans_labels

print("Cluster Distribution:")
print(df['KMeans_Cluster'].value_counts().sort_index())

## Section 7: Silhouette Score

In [ ]:
from sklearn.metrics import silhouette_score

sil_score = silhouette_score(X_scaled, kmeans_labels)
print("Silhouette Score:", round(sil_score, 4))

## Section 8: DBSCAN Clustering

In [ ]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(eps=1.5, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_scaled)

df['DBSCAN_Cluster'] = dbscan_labels

n_clusters_dbscan = len(set(dbscan_labels) - {-1})
n_noise = list(dbscan_labels).count(-1)

print("DBSCAN Clusters:", n_clusters_dbscan)
print("Noise Points:", n_noise)
print("\nDistribution:")
print(df['DBSCAN_Cluster'].value_counts().sort_index())

## Section 9: PCA and Visualization

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print("Explained Variance:", pca.explained_variance_ratio_)
print("Total Variance Explained:", round(sum(pca.explained_variance_ratio_) * 100, 2), "%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
for i in range(best_k):
    mask = kmeans_labels == i
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=colors[i], s=70, alpha=0.7,
                    edgecolors='white', linewidth=1,
                    label=f'Cluster {i} (n={mask.sum()})')

centroids_pca = pca.transform(kmeans_final.cluster_centers_)
axes[0].scatter(centroids_pca[:, 0], centroids_pca[:, 1],
                c='black', marker='X', s=200, edgecolors='gold',
                linewidths=2, label='Centroids')

axes[0].set_title('K-Means Clustering - PCA Projection')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

unique_dbscan = sorted(set(dbscan_labels))
dbscan_cmap = plt.cm.Set2(np.linspace(0, 1, max(len(unique_dbscan), 2)))
for idx, cid in enumerate(unique_dbscan):
    mask = dbscan_labels == cid
    label = 'Noise' if cid == -1 else f'Cluster {cid}'
    color = 'gray' if cid == -1 else dbscan_cmap[idx]
    alpha = 0.3 if cid == -1 else 0.7
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=[color], s=70, alpha=alpha,
                    edgecolors='white', linewidth=1,
                    label=f'{label} (n={mask.sum()})')

axes[1].set_title('DBSCAN Clustering - PCA Projection')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Section 10: Cluster Profiling

In [ ]:
feature_cols = ['child_mort', 'exports', 'health', 'imports', 'income',
                'inflation', 'life_expec', 'total_fer', 'gdpp']

cluster_profile = df.groupby('KMeans_Cluster')[feature_cols].mean().round(2)
print("Cluster Profiles:")
print(cluster_profile)

In [ ]:
cluster_profile_norm = (cluster_profile - cluster_profile.min()) / (cluster_profile.max() - cluster_profile.min())

plt.figure(figsize=(14, 5))
sns.heatmap(cluster_profile_norm, annot=cluster_profile.values, fmt='.1f',
            cmap='RdYlGn', linewidths=2, linecolor='white',
            cbar_kws={'label': 'Normalized Scale'})
plt.title('Cluster Profile Heatmap')
plt.ylabel('Cluster')
plt.xlabel('Feature')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
for cluster in range(best_k):
    countries = df[df['KMeans_Cluster'] == cluster]['country'].values
    print(f"\nCluster {cluster} ({len(countries)} countries):")
    print(', '.join(countries))

## Section 11: Random Forest Classification

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

y = kmeans_labels
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

rf_model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_pred)
rf_cv = cross_val_score(rf_model, X_scaled, y, cv=5)

print("Random Forest Results:")
print(f"Test Accuracy: {rf_accuracy:.4f}")
print(f"CV Accuracy: {rf_cv.mean():.4f} +/- {rf_cv.std():.4f}")
print("\nClassification Report:")
print(classification_report(y_test, rf_pred))

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 6))
bars = plt.barh(feature_importance['Feature'], feature_importance['Importance'],
                color=plt.cm.viridis(np.linspace(0.2, 0.9, len(feature_importance))),
                edgecolor='white', linewidth=1.5)
plt.title('Random Forest - Feature Importance')
plt.xlabel('Importance Score')
plt.ylabel('Feature')

for bar, val in zip(bars, feature_importance['Importance']):
    plt.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

## Section 12: XGBoost Classification

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    random_state=42, use_label_encoder=False, eval_metric='mlogloss'
)
xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)
xgb_accuracy = accuracy_score(y_test, xgb_pred)
xgb_cv = cross_val_score(xgb_model, X_scaled, y, cv=5)

print("XGBoost Results:")
print(f"Test Accuracy: {xgb_accuracy:.4f}")
print(f"CV Accuracy: {xgb_cv.mean():.4f} +/- {xgb_cv.std():.4f}")
print("\nClassification Report:")
print(classification_report(y_test, xgb_pred))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

cm_rf = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues',
            xticklabels=[f'Cluster {i}' for i in range(best_k)],
            yticklabels=[f'Cluster {i}' for i in range(best_k)],
            ax=axes[0], linewidths=2, linecolor='white')
axes[0].set_title('Random Forest - Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

cm_xgb = confusion_matrix(y_test, xgb_pred)
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Oranges',
            xticklabels=[f'Cluster {i}' for i in range(best_k)],
            yticklabels=[f'Cluster {i}' for i in range(best_k)],
            ax=axes[1], linewidths=2, linecolor='white')
axes[1].set_title('XGBoost - Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

## Section 13: Model Comparison

In [ ]:
print("Model Comparison:")
print(f"{'Model':<20} {'Test Acc':>10} {'CV Mean':>10}")
print("-" * 40)
print(f"{'Random Forest':<20} {rf_accuracy:>10.4f} {rf_cv.mean():>10.4f}")
print(f"{'XGBoost':<20} {xgb_accuracy:>10.4f} {xgb_cv.mean():>10.4f}")
print(f"\nK-Means Silhouette Score: {sil_score:.4f}")

In [ ]:
models = ['Random Forest', 'XGBoost']
test_acc = [rf_accuracy, xgb_accuracy]
cv_acc = [rf_cv.mean(), xgb_cv.mean()]

x = np.arange(len(models))
width = 0.3

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, test_acc, width, label='Test Accuracy', color='#2196F3', edgecolor='white', linewidth=2)
bars2 = ax.bar(x + width/2, cv_acc, width, label='CV Mean Accuracy', color='#FF9800', edgecolor='white', linewidth=2)

ax.set_title('Model Performance Comparison')
ax.set_ylabel('Accuracy Score')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0, 1.1)
ax.legend()
ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{bar.get_height():.3f}', ha='center', fontsize=11)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{bar.get_height():.3f}', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

## Section 14: Observations and Conclusions

**Observation 1 - High-Mortality Clusters:**
One cluster is dominated by countries with extremely high child mortality rates (averaging 90-100+ per 1,000 live births) and low life expectancy (around 55-62 years). Nations such as Haiti, Chad, Sierra Leone, Niger, Central African Republic, and Mali populate this segment. These countries also exhibit high total fertility rates (5-7 children per woman) and very low GDP per capita (under $1,000). The combination of high mortality, low income, and low health expenditure marks this group as critically underdeveloped.

**Observation 2 - Top-Tier Economic Zones:**
A distinct cluster comprises wealthy, developed nations with high GDP per capita (ranging from $20,000 to $105,000) and high life expectancy (78-83 years). Countries like Luxembourg, Norway, Switzerland, Singapore, United States, Australia, and Denmark belong to this group. These nations show very low child mortality (2-7 per 1,000), high health spending (8-18% of GDP), and low fertility rates (1.2-2.0), consistent with advanced healthcare systems and post-demographic-transition societies.

**Observation 3 - Low-Development and Transitioning Economies:**
A middle cluster captures transitioning and developing economies with moderate metrics across the board. Countries like India, Indonesia, Philippines, Egypt, Bolivia, and Vietnam appear here, with moderate child mortality (20-60 per 1,000), moderate GDP per capita ($1,000-$10,000), and life expectancy around 65-75 years. These nations are in various stages of economic development, with some showing signs of rapid growth while others face structural challenges.

**Observation 4 - Inflation Volatility in Resource-Dependent Economies:**
Several countries within the lower-development clusters display extremely high inflation (>15%), notably Nigeria (104%), Mongolia (39.2%), Venezuela (45.9%), and Angola (22.4%). These are often resource-dependent economies where commodity price fluctuations drive macroeconomic instability.

**Observation 5 - Health Expenditure as a Cluster Discriminator:**
Health expenditure (as % of GDP) serves as a strong discriminating feature across clusters. Developed nations consistently invest 8-18% of GDP in health, correlating with low child mortality and high life expectancy. In contrast, the underdeveloped cluster averages only 3-5% health spending. The Random Forest feature importance analysis confirms that gdpp, child_mort, and income are the top-3 most important features for cluster classification.